# Chapter 4 上手代码

## 环境准备

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict
import re
import json
import datetime
import ast

In [2]:
load_dotenv()

True

In [ ]:
class HelloAgentsLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None

--- 调用LLM ---
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
好的，这里为您提供两种 Python 快速排序的实现方式：

### 方法一：Pythonic 风格（简洁易懂）
这种写法利用 Python 的列表推导式，代码非常简短，适合理解快速排序的核心思想（分治法）。

```python
def quick_sort(arr):
    # 递归终止条件：如果数组为空或只有一个元素，则直接返回
    if len(arr) <= 1:
        return arr
    
    # 选取基准值 (这里选中间的元素)
    pivot = arr[len(arr) // 2]
    
    # 分割数组
    # left: 小于基准值的元素
    # middle: 等于基准值的元素
    # right: 大于基准值的元素
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    
    # 递归排序并合并
    return quick_sort(left) + middle + quick_sort(right)

# 测试代码
if __name__ == "__main__":
    data = [3, 6, 8, 10, 1, 2, 1]
    sorted_data = quick_sort(data)
    print("排序结果:", sorted_data)
```

---

### 方法二：原地排序（标准算法实现）
这种写法不需要创建新的列表，直接在原数组上进行交换操作，因此空间复杂度更低（$O(\log n)$），是教科书和面试中标准的快速排序写法。

```python
def partition(arr, low, high):
    """分区函数：将小于基准值的元素放到左边，大于的放到右边"""
    pivot = arr[high]  # 选择最后一个元素作为基准
    i = low - 1        # i 指向小于基准值的区域的末尾
    

In [ ]:
# --- 客户端使用示例 ---
if __name__ == '__main__':
    try:
        llmClient = HelloAgentsLLM()
        
        exampleMessages = [
            {"role": "system", "content": "You are a helpful assistant that writes Python code."},
            {"role": "user", "content": "写一个快速排序算法"}
        ]
        
        print("--- 调用LLM ---")
        responseText = llmClient.think(exampleMessages)
        if responseText:
            print("\n\n--- 完整模型响应 ---")
            print(responseText)

    except ValueError as e:
        print(e)

## ReAct

### 搜索工具

In [5]:
from tavily import TavilyClient
from typing import Any

# ── 1. 搜索工具核心逻辑 ──────────────────────────────────────
def search(query: str) -> str:
    """
    一个基于 Tavily 的网页搜索工具。
    优先返回直接答案，否则返回前 3 条结果摘要。
    """
    print(f"🔍 正在执行 [Tavily] 网页搜索: {query}")
    try:
        api_key = os.getenv("TAVILY_API_KEY")
        if not api_key:
            return "错误: TAVILY_API_KEY 未在 .env 文件中配置。"

        client = TavilyClient(api_key=api_key)
        response = client.search(query, max_results=3)

        # 优先返回直接答案
        if response.get("answer"):
            return response["answer"]

        # 退而求其次，返回前 3 条结果摘要
        results = response.get("results", [])
        if results:
            snippets = [
                f"[{i+1}] {r.get('title', '')}\n{r.get('content', '')}"
                for i, r in enumerate(results)
            ]
            return "\n\n".join(snippets)

        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"


# ── 2. 通用工具执行器 ─────────────────────────────────────────
class ToolExecutor:
    """负责注册和执行工具的管理器。"""

    def __init__(self):
        self.tools: dict[str, dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        if name in self.tools:
            print(f"警告: 工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"✅ 工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        return "\n".join(
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        )


In [6]:
# 初始化并测试
toolExecutor = ToolExecutor()

search_description = (
    "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
)
toolExecutor.registerTool("Search", search_description, search)

print("\n--- 可用的工具 ---")
print(toolExecutor.getAvailableTools())

print("\n--- 测试搜索 ---")
result = search("华为最新手机型号")
print(result)

✅ 工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 测试搜索 ---
🔍 正在执行 [Tavily] 网页搜索: 华为最新手机型号
[1] 【最新华为手机】最新华为手机报价及图片大全 - ZOL报价- 中关村在线
Redmi Turbo 5 MAX   Neo8 ƻ iPhone 17 Pro Max iQOO 15 Ultra Ϊ.  2TB 1TB 512GB 256GB 128GB 64GB.  24GB 18GB 16GB 12GB 10GB 8GB 6GB. * ### Ϊnova 15256GB ཹκӰ8020оƬAI. * ### ΪMate X712GB/256GB 9030 ProоƬɿ۵ܹڶӰ. * ### Ϊnova 15512GB ཹκӰ8020оƬAI. * ### ΪMate X7 ذ棨16GB/1TB 9030 ProоƬɿ۵ܹڶӰ. * ### ΪMate X7 ذ棨16GB/512GB 9030 ProоƬɿ۵ܹڶӰ. * ### ΪMate X7 ذ棨16GB/1TB/дװ 9030 ProоƬɿ۵ܹڶӰ. * ### HUAWEI Mate 80(12GB/256GB) 9020оƬڶӰ񣬺AI9020оƬڶӰ񣬺AI"). * ### HUAWEI Mate 80 Pro Max(16GB/512GB) 9030 ProоƬȫܹ͸9030 ProоƬȫܹ͸"). * ### HUAWEI Mate 80 Pro(12GB/512GB) 9030ϵоƬڶӰ񣬺AI9030ϵоƬڶӰ񣬺AI"). * ### HUAWEI Mate 80(12GB/512GB) 9020оƬڶӰ񣬺AI9020оƬڶӰ񣬺AI"). * ### HUAWEI Mate 80(16GB/512GB) 9020оƬڶӰ񣬺AI9020оƬڶӰ񣬺AI"). * ### HUAWEI Mate 80 RS Ƿʦ(20GB/512GB) ɫܹڶӰɫܹڶӰ"). * ### HUAWEI Mate 80 Pro Max(16GB/1TB) 9030 ProоƬȫܹ͸9030 ProоƬȫܹ͸"). * ### HUAWEI Mate 

### 获取系统时间工具

In [7]:
from datetime import datetime

# 定义工具
def get_current_date(format_str="%Y-%m-%d %H:%M:%S"):
	now = datetime.now()
	if not format_str:                          # ← 加这个判断
		format_str = "%Y-%m-%d %H:%M:%S"
	return now.strftime(format_str)


# 测试

get_current_date_description = (
	"获取当前系统时间。当你需要使用当前时间时，应使用此工具"
	)
toolExecutor.registerTool("get_current_date", get_current_date_description, get_current_date)

print("\n--- 可用的工具 ---")
print(toolExecutor.getAvailableTools())

print("\n--- 测试系统时间 ---")
result = get_current_date()
print(result)

✅ 工具 'get_current_date' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。
- get_current_date: 获取当前系统时间。当你需要使用当前时间时，应使用此工具

--- 测试系统时间 ---
2026-02-27 17:54:18


### ReAct 架构

In [8]:
import json
# ── 1. 提示词模板 ─────────────────────────────────────────────
REACT_SYSTEM_PROMPT = """你是一个能够调用外部工具的智能助手。
可用工具:
{tools}
每次回应时，你必须严格输出以下 JSON 格式（包裹在 ```json 代码块中）:
```json
{{
  "thought": "你的思考过程",
  "action": "工具名称 或 Finish",
  "action_input": "工具的输入 或 最终答案"
}}
"""

REACT_USER_TEMPLATE = """问题: {question}
历史记录: {history}
请继续:"""

In [9]:
class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps

    def run(self, question: str) -> str:
        history = []

        for step in range(1, self.max_steps + 1):
            print(f"\n{'='*45}")
            print(f"  Step {step} / {self.max_steps}")
            print(f"{'='*45}")

            # ① 格式化提示词
            history_text = "\n".join(history) if history else "（暂无）"
            user_prompt = REACT_USER_TEMPLATE.format(
                question=question,
                history=history_text,
            )
            messages = [
                {"role": "system", "content": REACT_SYSTEM_PROMPT.format(
                    tools=self.tool_executor.getAvailableTools()
                )},
                {"role": "user", "content": user_prompt},
            ]

            # ② 调用 LLM
            response = self.llm_client.think(messages=messages)
            if not response:
                raise RuntimeError("LLM 未返回任何内容。")

            # ③ 解析 JSON
            parsed = self._parse_json(response)
            thought       = parsed.get("thought", "")
            action        = parsed.get("action", "")
            action_input  = parsed.get("action_input", "")

            print(f"\n🤔 Thought: {thought}")
            print(f"🎬 Action:  {action}[{action_input}]")

            # ④ 判断是否完成
            if action == "Finish":
                print(f"\n🎉 最终答案:\n{action_input}")
                return action_input

            # ⑤ 调用工具
            tool_func = self.tool_executor.getTool(action)
            if not tool_func:
                raise RuntimeError(f"工具 '{action}' 不存在，请检查工具注册。")

            print(f"[DEBUG] tool_func={action}, action_input='{action_input}'")
            observation = tool_func(action_input)
            

            # ⑥ 追加历史
            history.append(f"Thought: {thought}")
            history.append(f"Action: {action}[{action_input}]")
            history.append(f"Observation: {observation}")

        print("\n⚠️  已达到最大循环次数，未能得出最终答案。")
        return None

    def _parse_json(self, text: str) -> dict:
        """从 LLM 响应中提取并解析 JSON 代码块。"""
        # 优先从 ```json ... ``` 中提取
        match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
        raw = match.group(1) if match else text.strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise RuntimeError(f"JSON 解析失败: {e}\n原始响应:\n{text}")


In [39]:
llmClient = HelloAgentsLLM()

agent = ReActAgent(llm_client=llmClient, tool_executor=toolExecutor)

agent.run("华为最新的手机是哪一款？它的主要卖点是什么？")



  Step 1 / 5
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
```json
{
  "thought": "用户询问华为最新的手机型号及其卖点。由于手机市场更新迅速，我需要联网搜索最新的信息来确认当前华为最新发布的手机型号。",
  "action": "Search",
  "action_input": "华为最新手机型号"
}
```

🤔 Thought: 用户询问华为最新的手机型号及其卖点。由于手机市场更新迅速，我需要联网搜索最新的信息来确认当前华为最新发布的手机型号。
🎬 Action:  Search[华为最新手机型号]
[DEBUG] tool_func=Search, action_input='华为最新手机型号'
🔍 正在执行 [Tavily] 网页搜索: 华为最新手机型号

  Step 2 / 5
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
```json
{
  "thought": "上一步的搜索结果显示了一些非常可疑的信息，例如“2026最新华为手机大全”以及“Mate 80”、“Pura 80”等型号。这很可能是搜索引擎返回了虚构的未来列表或错误的SEO内容。为了准确回答用户关于“最新”手机的问题，我首先需要确认当前的真实日期，然后根据日期去搜索真实已发布的最新机型（例如Mate系列或Pura系列的现状）。",
  "action": "get_current_date",
  "action_input": ""
}
```

🤔 Thought: 上一步的搜索结果显示了一些非常可疑的信息，例如“2026最新华为手机大全”以及“Mate 80”、“Pura 80”等型号。这很可能是搜索引擎返回了虚构的未来列表或错误的SEO内容。为了准确回答用户关于“最新”手机的问题，我首先需要确认当前的真实日期，然后根据日期去搜索真实已发布的最新机型（例如Mate系列或Pura系列的现状）。
🎬 Action:  get_current_date[]
[DEBUG] tool_func=get_current_date, action_input=''

  Step 3 / 5
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
```json
{
  "thought": "当前时

'华为目前最新的手机是 **HUAWEI Mate 80 系列**（于2025年11月25日发布）。\n\n其主要卖点包括：\n1. **屏幕技术突破**：首发“超透亮灵珑屏”，采用双层OLED架构，亮度、功耗、寿命三重飞跃，1% APL峰值亮度可达8000nits，并具备智能护眼功能。\n2. **影像系统升级**：搭载首发“第二代红枫影像系统”，大幅提升复杂光源下的色彩还原度，支持长焦微距慢动作及一次拍摄多重效果。\n3. **通信与户外功能**：首发700MHz无网应急通信，支持北斗卫星天气查询，并带来“户外探索模式”，具备路线信号预测及极限续航模式（可达14天）。\n4. **性能与设计**：Mate 80 Pro Max采用全金属设计与首创金属光绘微纹工艺；整机性能较上一代提升42%，支持实时光线追踪。\n5. **AI智慧体验**：全新升级的小艺支持自主学习、隔空传送、AI通话助手等功能。'

### Plan-and-Solve 架构

In [10]:
# ── 提示词模板 ────────────────────────────────────────────────

PLANNER_PROMPT = """你是一个任务分解专家。请将以下问题拆解成若干个独立的、按顺序执行的子步骤。

问题: {question}

要求:
- 每个步骤必须是一个独立的可执行子任务
- 步骤之间有明确的逻辑顺序
- 用 Python 列表格式输出，每个元素是一个步骤描述字符串

请将你的计划严格包裹在如下代码块中输出:
```python
["步骤1", "步骤2", ...]
"""

EXECUTOR_PROMPT = """你是一个任务执行专家。请根据下列信息，专注完成"当前步骤"，只输出该步骤的结果，不要附加解释。

原始问题
{question}

完整计划
{plan}

已完成步骤和结果
{history}

当前步骤
{current_step}

请直接输出当前步骤的结果:"""

In [15]:
class Planner: 
    def __init__(self, llm_client: HelloAgentsLLM, max_retries: int = 3): 
        self.llm_client = llm_client
        self.max_retries = max_retries

    def plan(self, question: str) -> list[str]:
        prompt = PLANNER_PROMPT.format(question=question)
        messages = [{"role": "user", "content": prompt}]
        for attempt in range(1, self.max_retries + 1):
            print(f"\n📋 正在生成计划（第 {attempt} 次尝试）...")
            response = self.llm_client.think(messages=messages)
            try:
                plan = self._parse_plan(response)
                print(f"\n✅ 计划生成成功，共 {len(plan)} 个步骤:")
                for i, step in enumerate(plan, 1):
                    print(f"  {i}. {step}")
                return plan
            except (ValueError, SyntaxError) as e:
                print(f"⚠️  第 {attempt} 次解析失败: {e}")
                if attempt == self.max_retries:
                    raise RuntimeError(f"规划失败：连续 {self.max_retries} 次无法解析有效计划。") from e
    
    def _parse_plan(self, text: str) -> list[str]:
        """从 LLM 响应中提取 Python 列表。"""
        match = re.search(r"```python\s*(\[.*?\])\s*```", text, re.DOTALL)
        raw = match.group(1) if match else text.strip()
        result = ast.literal_eval(raw)
        if not isinstance(result, list) or not result:
            raise ValueError("解析结果不是有效的非空列表")
        return result

In [16]:
class Executor: 
    def __init__(self, llm_client: HelloAgentsLLM): 
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        history_parts = []
        print(f"\n{'='*45}")
        print("  开始执行计划")
        print(f"{'='*45}")
        last_result = ""
        for i, step in enumerate(plan, 1):
            print(f"\n→ 步骤 {i}/{len(plan)}: {step}")
            history_text = "\n".join(history_parts) if history_parts else "（暂无）"
            prompt = EXECUTOR_PROMPT.format(
                question=question,
                plan="\n".join(f"{j+1}. {s}" for j, s in enumerate(plan)),
                history=history_text,
                current_step=step,
            )
            messages = [{"role": "user", "content": prompt}]
            last_result = self.llm_client.think(messages=messages) or ""
            print(f"✅ 结果: {last_result}")
            history_parts.append(f"步骤 {i}（{step}）→ {last_result}")
        return last_result

In [17]:
class PlanAndSolveAgent: 
    def __init__(self, llm_client: HelloAgentsLLM): 
        self.planner = Planner(llm_client) 
        self.executor = Executor(llm_client)

    def run(self, question: str) -> str:
        print(f"\n问题: {question}\n")
        plan = self.planner.plan(question)
        final_answer = self.executor.execute(question, plan)
        print(f"\n{'='*45}")
        print(f"🎉 最终答案: {final_answer}")
        print(f"{'='*45}")
        return final_answer

In [19]:
ps_agent = PlanAndSolveAgent(llm_client=HelloAgentsLLM())
ps_agent.run(
    "如何把大象装进冰箱？"
)


问题: 如何把大象装进冰箱？


📋 正在生成计划（第 1 次尝试）...
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
```python
["打开冰箱门", "把大象放进去", "关上冰箱门"]
```

✅ 计划生成成功，共 3 个步骤:
  1. 打开冰箱门
  2. 把大象放进去
  3. 关上冰箱门

  开始执行计划

→ 步骤 1/3: 打开冰箱门
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
冰箱门已打开。
✅ 结果: 冰箱门已打开。

→ 步骤 2/3: 把大象放进去
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
大象已放入冰箱。
✅ 结果: 大象已放入冰箱。

→ 步骤 3/3: 关上冰箱门
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
冰箱门已关上。
✅ 结果: 冰箱门已关上。

🎉 最终答案: 冰箱门已关上。


'冰箱门已关上。'

## Reflection 架构

In [20]:
# ── 提示词模板（三个角色，高度特质化）────────────────────────────

INITIAL_PROMPT = """
你是一位资深的 Python 程序员。请根据以下要求，编写一个 Python 函数。
代码必须包含完整的函数签名、文档字符串，并遵循 PEP 8 规范。

要求: {task}

请直接输出代码，不要包含任何额外解释。
"""

REFLECT_PROMPT = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码性能有极致的要求。
你的任务是审查以下 Python 代码，并专注于找出其在**算法效率**上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}

请分析该代码的时间复杂度，思考是否存在算法上更优的解决方案。

如果存在，请清晰指出瓶颈，并提出具体的改进算法建议。
如果代码在算法层面已达到最优，才能回答"无需改进"。
请直接输出你的反馈，不要附加任何额外解释。 """

REFINE_PROMPT = """ 你是一位资深的 Python 程序员。你正在根据评审专家的反馈来优化你的代码。
原始任务:
{task}

你上一轮的代码:
{last_code}

评审员的反馈:
{feedback}

请根据反馈生成优化后的新版本代码。 代码必须包含完整的函数签名、文档字符串，并遵循 PEP 8 规范。 请直接输出优化后的代码，不要包含任何额外解释。 """

In [21]:
class Memory: 
    """存储执行与反思的完整历史轨迹。"""
    def __init__(self):
        self.records: list[dict] = []
    def add_record(self, record_type: str, content: str):
        """添加一条记录，record_type 为 'execution' 或 'reflection'。"""
        self.records.append({"type": record_type, "content": content})
        print(f"📝 记忆已更新：新增一条 '{record_type}' 记录（当前共 {len(self.records)} 条）。")
    def get_last_execution(self) -> str | None:
        """返回最近一次执行结果。"""
        for record in reversed(self.records):
            if record["type"] == "execution":
                return record["content"]
        return None
    def get_trajectory(self) -> str:
        """将所有历史记录序列化为可读字符串，供提示词使用。"""
        parts = []
        for record in self.records:
            if record["type"] == "execution":
                parts.append(f"--- 上一轮代码 ---\n{record['content']}")
            elif record["type"] == "reflection":
                parts.append(f"--- 评审反馈 ---\n{record['content']}")
        return "\n\n".join(parts)

In [22]:
class ReflectionAgent:

    def __init__(self, llm_client, max_iterations: int = 3):
        self.llm_client = llm_client
        self.max_iterations = max_iterations
    def run(self, task: str) -> str:
        print(f"\n{'='*45}\n  开始处理任务\n{'='*45}")
        print(f"任务: {task}\n")
        memory = Memory()  # 每次 run 重置记忆
        # ① 初始执行
        print("--- 正在生成初始代码 ---")
        initial_code = self._call_llm(INITIAL_PROMPT.format(task=task))
        memory.add_record("execution", initial_code)
        # ② 反思-优化循环
        for i in range(1, self.max_iterations + 1):
            print(f"\n{'='*45}\n  第 {i}/{self.max_iterations} 轮迭代\n{'='*45}")
            # a. 审查器反思
            print("\n→ 正在反思...")
            last_code = memory.get_last_execution()
            feedback = self._call_llm(REFLECT_PROMPT.format(task=task, code=last_code))
            memory.add_record("reflection", feedback)
            # b. 检查停止条件
            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已达到算法最优，任务完成。")
                break
            # c. 执行器优化
            print("\n→ 正在优化...")
            refined_code = self._call_llm(REFINE_PROMPT.format(
                task=task,
                last_code=last_code,
                feedback=feedback,
            ))
            memory.add_record("execution", refined_code)
        # ③ 输出最终结果
        final_code = memory.get_last_execution()
        print(f"\n{'='*45}\n🎉 最终代码:\n```python\n{final_code}\n```\n{'='*45}")
        return final_code
    def _call_llm(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.llm_client.think(messages=messages) or ""

In [23]:
reflection_agent = ReflectionAgent(llm_client=HelloAgentsLLM(), max_iterations=3)

reflection_agent.run("编写一个 Python 函数，找出1到n之间所有的素数（prime numbers）。")



  开始处理任务
任务: 编写一个 Python 函数，找出1到n之间所有的素数（prime numbers）。

--- 正在生成初始代码 ---
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
```python
def find_primes(n: int) -> list[int]:
    """
    找出1到n之间所有的素数。

    该函数使用埃拉托斯特尼筛法 来高效地筛选素数。

    Args:
        n (int): 查找素数的上限（包含n）。

    Returns:
        list[int]: 包含1到n之间所有素数的列表，按升序排列。

    Raises:
        ValueError: 如果n小于1。
    """
    if n < 1:
        raise ValueError("输入参数n必须大于等于1。")
    if n == 1:
        return []

    # 初始化标记列表，索引代表数值，True表示可能是素数
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False

    # 只需遍历到 sqrt(n)
    for i in range(2, int(n**0.5) + 1):
        if is_prime[i]:
            # 将 i 的倍数标记为非素数，从 i*i 开始可以优化性能
            for j in range(i * i, n + 1, i):
                is_prime[j] = False

    # 收集所有标记为 True 的索引即为素数
    return [num for num, prime in enumerate(is_prime) if prime]
```
📝 记忆已更新：新增一条 'execution' 记录（当前共 1 条）。

  第 1/3 轮迭代

→ 正在反思...
🧠 正在调用 glm-5 模型...
✅ 大语言模型响应成功:
### 代码审查报告

#### 1. 算法效率瓶颈分析

当前代码实现的是标准的**埃拉托斯特尼筛法

'```python\ndef find_primes(n: int) -> list[int]:\n    """\n    使用欧拉筛 找出1到n之间所有的素数。\n\n    相比于埃拉托斯特尼筛法，欧拉筛保证了每个合数仅被其最小质因子标记一次，\n    从而将时间复杂度优化至线性级别 O(N)。\n\n    Args:\n        n (int): 查找素数的上限（包含n）。\n\n    Returns:\n        list[int]: 包含1到n之间所有素数的列表，按升序排列。\n\n    Raises:\n        ValueError: 如果n小于1。\n    """\n    if n < 1:\n        raise ValueError("输入参数n必须大于等于1。")\n    if n == 1:\n        return []\n\n    # is_prime 用于标记是否为素数\n    is_prime = [True] * (n + 1)\n    is_prime[0] = is_prime[1] = False\n\n    # primes 列表用于按顺序收集素数\n    primes = []\n\n    for i in range(2, n + 1):\n        if is_prime[i]:\n            primes.append(i)\n\n        # 用当前数 i 和已知素数 p 组合筛选合数 i*p\n        for p in primes:\n            # 如果合数超出范围，停止筛选\n            if i * p > n:\n                break\n\n            is_prime[i * p] = False\n\n            # 核心优化点：\n            # 如果 i 是 p 的倍数，说明 i * p 已经被 p (最小质因子) 筛过了。\n            # 此时必须停止，防止重复标记，保证 O(N) 复杂度。\n            if i % p == 0:\n                break\n\n    r